# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: FAIR² Dataset Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. The FAIR² dataset contains demographic, clinicopathological, anatomical, and molecular characteristics of 77 cancer survivors with second primary colorectal cancer.

### Dataset Source
- **Croissant Schema URL:** https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
- **Citation:** Liu, Y, Duan, X, Yang, S, Zhang, Y and Han, S 2026

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records from the FAIR² Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata object attributes
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Number of authors: {len(dataset.metadata.author) if hasattr(dataset.metadata, 'author') else 'N/A'}")
print(f"Published: {getattr(dataset.metadata, 'datePublished', 'N/A')}")
print(f"Citation: {getattr(dataset.metadata, 'citeAs', 'N/A')}")

## 2. Data Overview
Review available record sets and their schema structure. Each entity is referenced by its unique `@id` (as required for Croissant datasets).

**Find all record sets and view their fields (columns) and their canonical `@id`s.**

In [ ]:
# Discover record sets in the dataset by their @id
print("Available record sets (by @id):")

record_sets = dataset.record_sets
for record_set in record_sets:
    print(f"- {record_set['@id']} (name: {record_set.get('name', '<no name>')})")

# Example: List fields for each record set
for record_set in record_sets:
    print(f"\nRecord set: {record_set['@id']} (name: {record_set.get('name', '<no name>')})")
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        # Print field @id and column @id (if defined)
        field_id = field.get('@id', '<no @id>')
        field_name = field.get('name', '<no name>')
        data_type = field.get('dataType', '<no dataType>')
        column = field.get('column', {})
        if isinstance(column, dict):
            column_id = column.get('@id', '<no column @id>')
        elif isinstance(column, list) and column and isinstance(column[0], dict):
            column_id = column[0].get('@id', '<no column @id>')
        else:
            column_id = '<no column>'
        print(f"  - Field: {field_id} (name: {field_name}, type: {data_type}, column: {column_id})")

## 3. Data Extraction
Load the tabular data from the main record set into a DataFrame for analysis.

**Note**: All record sets and fields are referenced strictly by their `@id` (not name), as per best Croissant practices.

Below, we load one or more record sets by their IDs and preview their structure.

In [ ]:
# Collect all record set @ids
record_set_ids = [r['@id'] for r in dataset.record_sets]
print('Record set @ids:', record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nLoading records from: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"DataFrame columns for {record_set_id}: {dataframes[record_set_id].columns.tolist()}")
        display(dataframes[record_set_id].head())
    else:
        print(f"No records found for {record_set_id}")

# For later reference, pick the main tabular record set by its @id
main_record_set_id = record_set_ids[0] if record_set_ids else None
print('Main record set for EDA:', main_record_set_id)

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, groupings, etc.

**All columns and fields must be referenced via their `@id`.**

**Example steps:**
- Select a numeric field
- Filter out records based on a threshold
- Normalize the numeric column
- Group by a categorical variable

Adjust the field `@id`s below based on the record set overview printed before.

In [ ]:
# --- Identify numeric and categorical fields by their @id (replace with actual IDs from overview step) --- #
main_df = dataframes[main_record_set_id]

# Print available columns (these are the field @ids)
print('Columns available in main record set:', main_df.columns.tolist())

# Example: Assume one field represents "Age at second CRC diagnosis" and another is "Sex" (replace with actual @ids)
numeric_field_id = next((col for col in main_df.columns if 'Age' in col or 'age' in col), main_df.columns[0])
group_field_id = next((col for col in main_df.columns if ('Sex' in col or 'sex' in col or 'Gender' in col or 'gender' in col)), None)

print(f"Using numeric field: {numeric_field_id}")
if group_field_id:
    print(f"Categorical grouping field: {group_field_id}")

# Filter for ages > 50 as an EDA example
if main_df[numeric_field_id].dtype in [int, float, 'int64', 'float64']:
    threshold = 50
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
else:
    # Try to convert if not already numeric (fallback)
    filtered_df = main_df.copy()
    filtered_df[numeric_field_id] = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
    threshold = 50
    filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
    filtered_df = filtered_df.dropna(subset=[numeric_field_id])

print(f"Filtered records where {numeric_field_id} > {threshold} (first 5):")
display(filtered_df.head())

# Normalize the numeric field
mean = filtered_df[numeric_field_id].mean()
std = filtered_df[numeric_field_id].std()
filtered_df[f'{numeric_field_id}_normalized'] = (filtered_df[numeric_field_id] - mean) / std

print(f"Normalized '{numeric_field_id}' for filtered records (first 5):")
display(filtered_df[[numeric_field_id, f'{numeric_field_id}_normalized']].head())

# Group by category if available
if group_field_id and group_field_id in filtered_df.columns:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
    display(grouped)
else:
    print('No suitable group field found for demonstration.')

## 5. Visualization

Visualize distributions or relationships in the dataset using field `@id`s.

**Example Visualizations:**
- Histogram of a numeric variable (e.g., Age at diagnosis)
- Boxplot or barplot grouped by a categorical attribute (e.g., Sex)
- Scatter plot if suitable fields are available


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set styles for clarity
sns.set(style="whitegrid", palette="muted")

if not filtered_df.empty:
    # Histogram of the normalized numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[f'{numeric_field_id}_normalized'], bins=10, kde=True)
    plt.title(f'Histogram of Normalized "{numeric_field_id}"')
    plt.xlabel(f'{numeric_field_id} (normalized)')
    plt.show()

    # Boxplot by group if group field exists
    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(6, 4))
        sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id])
        plt.title(f'"{numeric_field_id}" by "{group_field_id}"')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No data to visualize after filtering.")

## 6. Conclusion

In this notebook, we demonstrated:
- Loading comprehensive metadata and records from a Croissant FAIR² dataset using only `@id` references
- Overview of record set and field structure strictly by `@id` (best practice for reproducibility)
- Extraction and processing of tabular patient-level data
- Exploratory filtering, normalization, grouping, and visualization of key variables

**Next steps:**
- Extend the EDA to include additional variables and domain-specific questions
- Use other record sets as appropriate using their Croissant `@id`
- Apply ML preprocessing or modeling using the prepared DataFrames